# The file will build a simple linear model as the baseline, and several different General Linear Model(GLM).

In [1]:
# import needed libraries
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import glm
from statsmodels.api import families
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

### 1. Load modelling data

In [2]:
modelling_data = pd.read_csv('../data/curated/final_data/modelling_data_2023.csv')
modelling_data

,personal_income,pop_density,offence_count,log_school_distance,log_station_distance,log_hospital_distance,log_mall_distance,log_park_distance,log_CBD_distance,num_bedroom,num_bathroom,rental_price
0,72706.625309,965.216952,17.333333,0.640804,2.475412,1.906040,1.239505,1.345502,4.578730,4,2,575.0
1,72706.625309,1746.760558,17.333333,-0.348952,1.681479,1.438925,1.052926,1.716843,4.559835,4,2,560.0
2,72706.625309,965.216952,17.333333,0.095820,2.187960,1.847653,1.673884,1.622239,4.586220,2,2,490.0
3,72706.625309,965.216952,17.333333,-0.393160,2.092679,1.736704,0.793410,1.183953,4.578058,4,2,540.0
4,70490.898811,151.921192,31.000000,0.799813,2.315778,2.017863,1.375422,0.115940,4.602687,4,2,520.0
...,...,...,...,...,...,...,...,...,...,...,...,...
8214,100462.815420,1773.237842,25.000000,-2.228826,0.138021,0.374363,0.344505,1.026702,3.826546,2,1,630.0
8215,100462.815420,1773.237842,25.000000,-1.427862,0.754430,1.122437,0.873918,1.320736,3.823240,4,3,730.0
8216,100462.815420,1773.237842,25.000000,-0.199202,1.034856,1.025788,0.821525,1.484705,3.805227,3,1,450.0
8217,100462.815420,1773.237842,25.000000,-1.237029,0.614537,0.623725,0.746158,1.316399,3.810149,1,1,300.0


In [3]:
# collect all features
final_features = modelling_data.columns
features_list = final_features.tolist()
features_list.remove('rental_price')
features_list

['personal_income',
 'pop_density',
 'offence_count',
 'log_school_distance',
 'log_station_distance',
 'log_hospital_distance',
 'log_mall_distance',
 'log_park_distance',
 'log_CBD_distance',
 'num_bedroom',
 'num_bathroom']

### 2. Simple Linear Model

In [4]:
# split training set and test set
X_train, X_test, y_train, y_test = train_test_split(modelling_data[features_list], modelling_data['rental_price'], test_size = 0.1, random_state=123)

# initialize and fit a linear regression model
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

# make predictions and evaluate the model
y_pred = linear_model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print("MSE: {:.2f}".format(mse))
print("MAE: {:.2f}".format(mae))
print("r2 score: {:.2f}".format(r2))

MSE: 13107.11
MAE: 83.90
r2 score: 0.45


In [5]:
coefficients = linear_model.coef_
intercept = linear_model.intercept_

print("Coefficients:", coefficients)
print("Intercept:", intercept)

Coefficients: [ 2.33882707e-03  5.78788839e-03 -3.55001527e-01 -9.96891962e+00
 -2.34701016e+01 -2.24819808e+01 -1.58212786e+01 -1.94861273e+01
  1.84848423e+01  5.79417258e+01  8.14031455e+01]
Intercept: 63.833044510882644


### 3. General Linear Model

In [6]:
# define the formula for all GLMs
formula = "rental_price ~ personal_income * pop_density * offence_count + log_school_distance * log_station_distance * log_hospital_distance * log_mall_distance * log_park_distance * log_CBD_distance + num_bedroom + num_bathroom" # if features changed, remember to fix here!
formula

'rental_price ~ personal_income * pop_density * offence_count + log_school_distance * log_station_distance * log_hospital_distance * log_mall_distance * log_park_distance * log_CBD_distance + num_bedroom + num_bathroom'

#### a). GLM with Gamma family

In [7]:
# combine X_train and y_train as one DataFrame
train_data = X_train.copy()
train_data['rental_price'] = y_train

# build Gamma GLM
gamma_glm = glm(
    formula = formula,
    data = train_data,
    family = sm.families.Gamma()
)

# fit the model
gamma_fit = gamma_glm.fit()
# summary of the model
print(gamma_fit.summary())

/home/ximing/.local/lib/python3.10/site-packages/statsmodels/genmod/generalized_linear_model.py:307: DomainWarning: The InversePower link function does not respect the domain of the Gamma family.
  warnings.warn((f"The {type(family.link).__name__} link function "


                 Generalized Linear Model Regression Results                  
Dep. Variable:           rental_price   No. Observations:                 7397
Model:                            GLM   Df Residuals:                     7329
Model Family:                   Gamma   Df Model:                           67
Link Function:           InversePower   Scale:                        0.044455
Method:                          IRLS   Log-Likelihood:                -46413.
Date:                Wed, 04 Oct 2023   Deviance:                       417.99
Time:                        16:39:31   Pearson chi2:                     326.
No. Iterations:                   100   Pseudo R-squ. (CS):             0.4647
Covariance Type:            nonrobust                                         
                                                                                                                          coef    std err          z      P>|z|      [0.025      0.975]
--------------------------

In [8]:
# make predictions and evaluate the model
y_pred = gamma_fit.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print("MSE: {:.2f}".format(mse))
print("MAE: {:.2f}".format(mae))
print("r2 score: {:.2f}".format(r2))

MSE: 11780.72
MAE: 78.14
r2 score: 0.51


#### b). GLM with Poisson family

In [9]:
# build Gamma GLM
poisson_glm = glm(
    formula = formula,
    data = train_data,
    family = sm.families.Poisson()
)

# fit the model
poisson_fit = poisson_glm.fit()
# summary of the model
print(poisson_fit.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:           rental_price   No. Observations:                 7397
Model:                            GLM   Df Residuals:                     7329
Model Family:                 Poisson   Df Model:                           67
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -1.1941e+05
Date:                Wed, 04 Oct 2023   Deviance:                   1.7888e+05
Time:                        16:39:34   Pearson chi2:                 1.72e+05
No. Iterations:                    38   Pseudo R-squ. (CS):              1.000
Covariance Type:            nonrobust                                         
                                                                                                                          coef    std err          z      P>|z|      [0.025      0.975]
--------------------------

In [10]:
# make predictions and evaluate the model
y_pred = poisson_fit.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print("MSE: {:.2f}".format(mse))
print("MAE: {:.2f}".format(mae))
print("r2 score: {:.2f}".format(r2))

MSE: 11546.81
MAE: 77.29
r2 score: 0.52


#### c). GLM with Gaussian family

In [11]:
# build Gamma GLM
gaussian_glm = glm(
    formula=formula,
    data=train_data,
    family=sm.families.Gaussian()
)

# fit the model
gaussian_fit = gaussian_glm.fit()
# summary of the model
print(gaussian_fit.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:           rental_price   No. Observations:                 7397
Model:                            GLM   Df Residuals:                     7329
Model Family:                Gaussian   Df Model:                           67
Link Function:               Identity   Scale:                          13281.
Method:                          IRLS   Log-Likelihood:                -45576.
Date:                Wed, 04 Oct 2023   Deviance:                   9.7339e+07
Time:                        16:39:35   Pearson chi2:                 9.73e+07
No. Iterations:                     3   Pseudo R-squ. (CS):             0.5719
Covariance Type:            nonrobust                                         
                                                                                                                          coef    std err          z      P>|z|      [0.025      0.975]
--------------------------

In [12]:
# make predictions and evaluate the model
y_pred = gamma_fit.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print("MSE: {:.2f}".format(mse))
print("MAE: {:.2f}".format(mae))
print("r2 score: {:.2f}".format(r2))

MSE: 11780.72
MAE: 78.14
r2 score: 0.51


# Summary: Overall poor performance, GLM will not be used.